In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
os.listdir('/content/drive/MyDrive/Colab')

['placement_dataset.csv',
 'placement_predict_50k Dataset (1) (1).csv',
 'placement_predict_50k_adjusted.csv',
 'age_insurance.csv',
 'simple_placement_dataset.csv',
 'movies_dataset(1).csv',
 'WineQT.csv',
 'WineQT_Feature_Engineered.csv',
 'placement_predict_50k_adjusted (1).csv',
 'student_data.csv',
 'xgboost_lightgbm_shap_student_dataset.csv',
 'iris_dataset.csv',
 'breast_cancer_classification_dataset (1).csv',
 'mall_customers_clustering_dataset.csv',
 'digits_pca_tsne_umap_dataset.csv',
 'isolation_forest_dataset.csv',
 'socioeconomic_profiling_dataset.csv',
 'ml_cross_validation_calibration_dataset.csv',
 'hyperparameter_tuning_dataset.csv',
 'evaluation_threshold_calibration_dataset.csv',
 'student_performance.csv']

In [3]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Colab/placement_dataset.csv')

In [5]:
# ============================================================
# MODULAR ML PIPELINE + FASTAPI DEPLOYMENT
# Student Performance Prediction
# ============================================================

import os
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

from fastapi import FastAPI
from pydantic import BaseModel


# ============================================================
# 1. CONFIGURATION
# ============================================================

DATA_FILE = "/content/drive/MyDrive/Colab/student_performance.csv"
MODEL_FILE = "student_model.pkl"


# ============================================================
# 2. FEATURES
# ============================================================

FEATURES = [
    "study_hours",
    "attendance",
    "previous_score",
    "assignments_completed",
    "sleep_hours",
    "internet_access",
    "extracurricular"
]

TARGET = "pass"


# ============================================================
# 3. LOAD DATA
# ============================================================

def load_data():

    print("\nLoading dataset...")

    df = pd.read_csv(DATA_FILE)

    print("Dataset loaded successfully!")
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])

    return df


# ============================================================
# 4. PREPARE DATA
# ============================================================

def prepare_data(df):

    X = df[FEATURES]

    y = df[TARGET]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    return X_train, X_test, y_train, y_test


# ============================================================
# 5. CREATE MACHINE LEARNING PIPELINE
# ============================================================

def create_pipeline():

    pipeline = Pipeline([

        # Handle missing values
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),

        # Scale numerical values
        (
            "scaler",
            StandardScaler()
        ),

        # Machine Learning Model
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=100,
                random_state=42
            )
        )
    ])

    return pipeline


# ============================================================
# 6. TRAIN MODEL
# ============================================================

def train_model():

    print("\n" + "=" * 60)
    print("TRAINING MODEL")
    print("=" * 60)

    # Load dataset
    df = load_data()

    # Prepare data
    X_train, X_test, y_train, y_test = prepare_data(df)

    # Create pipeline
    pipeline = create_pipeline()

    # Train model
    pipeline.fit(X_train, y_train)

    # Make predictions
    predictions = pipeline.predict(X_test)

    # Calculate accuracy
    accuracy = accuracy_score(
        y_test,
        predictions
    )

    print("\nModel Accuracy:")
    print(accuracy)

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            predictions
        )
    )

    # Save trained model
    joblib.dump(
        pipeline,
        MODEL_FILE
    )

    print("\nModel saved successfully!")
    print("Model file:", MODEL_FILE)

    return pipeline


# ============================================================
# 7. LOAD TRAINED MODEL
# ============================================================

def load_model():

    if os.path.exists(MODEL_FILE):

        print("\nLoading existing model...")

        return joblib.load(MODEL_FILE)

    else:

        print("\nModel not found.")
        print("Training new model...")

        return train_model()


# ============================================================
# 8. FASTAPI APPLICATION
# ============================================================

app = FastAPI(
    title="Student Performance Prediction API",
    description="Machine Learning model deployed using FastAPI",
    version="1.0"
)


# ============================================================
# 9. INPUT DATA MODEL
# ============================================================

class StudentData(BaseModel):

    study_hours: float

    attendance: float

    previous_score: float

    assignments_completed: int

    sleep_hours: float

    internet_access: int

    extracurricular: int


# ============================================================
# 10. HOME API
# ============================================================

@app.get("/")
def home():

    return {
        "message": "Student Performance Prediction API is running",
        "status": "success"
    }


# ============================================================
# 11. PREDICTION API
# ============================================================

@app.post("/predict")
def predict(data: StudentData):

    # Load model
    model = load_model()

    # Convert input into dictionary
    student_data = {

        "study_hours": data.study_hours,

        "attendance": data.attendance,

        "previous_score": data.previous_score,

        "assignments_completed":
            data.assignments_completed,

        "sleep_hours": data.sleep_hours,

        "internet_access":
            data.internet_access,

        "extracurricular":
            data.extracurricular
    }

    # Convert dictionary into DataFrame
    input_data = pd.DataFrame(
        [student_data]
    )

    # Make prediction
    prediction = model.predict(
        input_data
    )[0]

    # Get probability
    probability = model.predict_proba(
        input_data
    )[0]

    # Convert prediction to result
    if prediction == 1:

        result = "PASS"

    else:

        result = "FAIL"

    return {

        "prediction": int(prediction),

        "result": result,

        "pass_probability":
            round(float(probability[1]), 4),

        "fail_probability":
            round(float(probability[0]), 4)
    }


# ============================================================
# 12. TRAIN MODEL WHEN FILE IS RUN DIRECTLY
# ============================================================

if __name__ == "__main__":

    print("\n" + "=" * 60)
    print("STUDENT PERFORMANCE ML PROJECT")
    print("=" * 60)

    # Train model
    train_model()

    print("\nTo start FastAPI, run:")
    print(
        "uvicorn app:app --reload"
    )


STUDENT PERFORMANCE ML PROJECT

TRAINING MODEL

Loading dataset...
Dataset loaded successfully!
Rows: 200
Columns: 9

Model Accuracy:
0.925

Classification Report:
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.95      0.97      0.96        38

    accuracy                           0.93        40
   macro avg       0.47      0.49      0.48        40
weighted avg       0.90      0.93      0.91        40


Model saved successfully!
Model file: student_model.pkl

To start FastAPI, run:
uvicorn app:app --reload
